# Praktikum 5: Notebook 3: Delta Lake ja lakehouse

**Moodul 5: Suurandmed ja pilvelahendused (edasijõudnud)**

**Hinnanguline aeg:** 20 minutit

**Eeldused:** Läbitud notebook 01 ja 02. `samples.nyctaxi.trips` on saadaval.

## Mida me õpime

Notebook 02 näitas, et Delta on Parquet + `_delta_log`. Nüüd vaatame, mida see logi annab:

1. ACID operatsioonid (INSERT, UPDATE, DELETE, MERGE)
2. Time travel: varasema versiooni lugemine
3. OPTIMIZE ja VACUUM
4. Medaljoni arhitektuur (bronze / silver / gold) praktikas


## 1. Delta tabeli loomine ja ACID operatsioonid

Alustame väikese tabeliga, mille saame ühe lahtrit korraga näha. Kasutame `samples.nyctaxi.trips` osa.

In [ ]:
import pyspark.sql.functions as F

# Loome Delta tabeli 500 sõiduga
(
    spark.table("samples.nyctaxi.trips")
    .limit(500)
    .withColumn("trip_id", F.monotonically_increasing_id())
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("trips_delta")
)

spark.table("trips_delta").show(5)

In [ ]:
# UPDATE: muuda 0-ga fare_amount-ga ridadele väärtus 2.50 (miinimum tariif)
spark.sql("UPDATE trips_delta SET fare_amount = 2.50 WHERE fare_amount < 2.50")

# Vaata ajalugu
spark.sql("DESCRIBE HISTORY trips_delta").select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

In [ ]:
# DELETE: kustuta read, kus trip_distance > 50 (ilmselt andmeviga)
spark.sql("DELETE FROM trips_delta WHERE trip_distance > 50")

spark.sql("DESCRIBE HISTORY trips_delta").select("version", "operation", "operationMetrics").show(truncate=False)

In [ ]:
# MERGE: upsert tüüpiline kasutus
# Loome väikese uue DataFrame'i, kus osa trip_id-sid on olemasolevad, osa uued
updates = spark.createDataFrame(
    [
        (0, 99.99),
        (1, 88.88),
        (999999, 123.45),  # uus rida
    ],
    "trip_id BIGINT, fare_amount DOUBLE",
)
updates.createOrReplaceTempView("updates_v")

# MÄRKUS: INSERT annab ainult trip_id ja fare_amount — muud veerud jäävad NULL.
# Päriskasutuses pakud tavaliselt kõik kohustuslikud veerud kaasa.
spark.sql("""
    MERGE INTO trips_delta t
    USING updates_v u
    ON t.trip_id = u.trip_id
    WHEN MATCHED THEN UPDATE SET t.fare_amount = u.fare_amount
    WHEN NOT MATCHED THEN INSERT (trip_id, fare_amount) VALUES (u.trip_id, u.fare_amount)
""")

spark.sql("DESCRIBE HISTORY trips_delta").select("version", "operation", "operationMetrics").show(truncate=False)

**Miks MERGE on oluline?** Klassikalisel data lake'il (Parquet ilma Delta'ta) ei ole upsert lihtne. Pead tegema:

1. Loe kogu vana tabel
2. Tee join uue andmega
3. Kirjuta kogu tabel uuesti

See on aeglane (mitte väga paralleelne) ja kergesti võivad tekkida vead. Delta MERGE teeb sama asja ühe transaktsioonina ja puudutab ainult muudetavaid faile.


Viide: [Delta MERGE INTO](https://docs.delta.io/latest/delta-update.html#upsert-into-a-table-using-merge)

## 2. Time travel

Iga `UPDATE`, `DELETE`, `MERGE`, `INSERT` tekitab uue versiooni. Sa saad lugeda mistahes varasemat versiooni.

In [ ]:
# Loendame ridu iga versiooni jaoks
history = spark.sql("DESCRIBE HISTORY trips_delta").select("version").orderBy("version").collect()

for row in history:
    v = row["version"]
    n = spark.read.format("delta").option("versionAsOf", v).table("trips_delta").count()
    print(f"Versioon {v}: {n} rida")

In [ ]:
# VERSION AS OF SQL sõnastuses
spark.sql("SELECT COUNT(*) AS rows_v0 FROM trips_delta VERSION AS OF 0").show()
spark.sql("SELECT COUNT(*) AS rows_latest FROM trips_delta").show()

Time travel on praktiliselt oluline kolme asja jaoks:

1. **Auditi jälg.** Kui keegi küsib "mis see väärtus eile oli", ei pea sul päev-päeva kaupa snapshoti ajalugu olema.
2. **Vea taastamine.** Kui vale MERGE kustutas andmeid, saad `RESTORE TABLE trips_delta TO VERSION AS OF N`.
3. **Reprodutseeritavad päringud.** Analüütiku aruande puhul saad viidata kindlale versioonile.

**NB!** Time travel ajaloo pikkust piirab `delta.deletedFileRetentionDuration` (vaikimisi 7 päeva). Pikem ajalugu tähendab, et vanu faile hoitakse kauem == suurem salvetusmahu kasutus.


Viide: [Delta time travel](https://docs.delta.io/latest/delta-batch.html#query-an-older-snapshot-of-a-table-time-travel)

## 3. OPTIMIZE ja VACUUM

Delta tabelisse korduvalt kirjutamine loob palju väikeseid faile. `OPTIMIZE` kompakteerib need suuremateks. `VACUUM` kustutab vanad, kasutuseta failid.

Free Edition'is on OPTIMIZE lubatud, aga kompaktimise efekti on meie väikeses tabelis raske näha.

In [ ]:
# OPTIMIZE: kompakteerib väikeseid faile
spark.sql("OPTIMIZE trips_delta").show(truncate=False)

In [ ]:
# Vaata uut versiooni ajaloos
spark.sql("DESCRIBE HISTORY trips_delta").select("version", "operation", "operationMetrics").show(truncate=False)

**VACUUM** kustutab failid, mida ei ole enam aktiivses Delta logis ja mis on vanemad kui `retentionHours`. Vaikimisi on see 7 päeva (168 tundi). Lühemat retentsiooni saab lubada ainult `SET spark.databricks.delta.retentionDurationCheck.enabled = false` järel. Free Edition'is me seda ei tee, sest see tähendab potentsiaalselt töös oleva päringu lõhkumist.

Praktikas jooksutatakse `OPTIMIZE` perioodiliselt (üks kord päevas või nädalas) scheduled job'ina, `VACUUM` harvem.


Viide: [OPTIMIZE ja VACUUM](https://docs.databricks.com/en/delta/optimize.html)

## 4. Medaljoni arhitektuur

Medaljoni arhitektuur on viis, kuidas lakehouse'is andmed läbi kihtide liiguvad:

```
  toorandmed                  puhastatud                 äriline kokkuvõte
  (raw files)                 (typed, valid)             (KPI, aggregates)
        
  [ BRONZE ]   ---ETL--->    [ SILVER ]   ---ETL--->    [ GOLD ]
        
  failid, 1:1                tüüpide parandus            äri mõõdikud
  allikaga,                  duplikaatide                päeva/kuu
  ajalugu alles              eemaldamine                 agregatsioonid
                             vigaste ridade             analüütikute
                             filtreerimine              jaoks
```

**Miks kolm kihti?**

- **Bronze** hoiab toorandmed muutmata. Kui silver kihis avastad bug'i, saad taastada ilma allikasse tagasi minekuta.
- **Silver** on "usaldusväärne" kiht: tüübid on õiged, duplikaadid eemaldatud, põhilised filtrid rakendatud. Data scientist'id töötavad tavaliselt siin.
- **Gold** on äri-tasandil mõõdikud. Dashboard'id loevad siit.

Iga kihti hoitakse Delta tabelitena, et ACID ja time travel töötaksid kogu ahelas.


Viide: [Medallion architecture](https://www.databricks.com/glossary/medallion-architecture)

### Mini-näide NYC Taxi andmetega

Võtame notebook 02 sample andmed ja laome need kolme kihti.

In [ ]:
yellow_path = "wasbs://nyctlc@azureopendatastorage.blob.core.windows.net/yellow"

# Bronze: loeme toorandmed sisse ja salvestame 1:1
# NB! Piirame ühe aastaga, et veenduda, et kogu sample pärineb sama skeemiga perioodist.
# NYC Taxi vanemates aastates (enne 2010) oli `paymentType` stringina ('CSH', 'CRD'),
# hiljem on see täisarv (1..6). 2016 on selgelt "uue" skeemi aasta.
bronze = (
    spark.read.parquet(yellow_path)
    .where(F.year("tpepPickupDateTime") == 2016)
    .limit(500_000)
)
bronze.write.mode("overwrite").format("delta").saveAsTable("taxi_bronze")

print("Bronze ridu:", spark.table("taxi_bronze").count())

In [ ]:
# Silver: tüüpide teisendus, vigaste ridade filtreerimine
# NB! Bronze pärineb Azure Open Datasets'ist, seal on veerunimed camelCase.
# Silver väljundis nimetame need ümber snake_case'iks (meie oma konventsioon).
silver = (
    spark.table("taxi_bronze")
    .where(F.col("fareAmount") > 0)
    .where(F.col("tripDistance") > 0)
    .where(F.col("tpepDropoffDateTime") > F.col("tpepPickupDateTime"))
    .select(
        F.col("tpepPickupDateTime").cast("timestamp").alias("pickup_ts"),
        F.col("tpepDropoffDateTime").cast("timestamp").alias("dropoff_ts"),
        F.col("passengerCount").cast("int").alias("passenger_count"),
        F.col("tripDistance").cast("double").alias("trip_distance"),
        F.col("fareAmount").cast("double").alias("fare_amount"),
        F.col("totalAmount").cast("double").alias("total_amount"),
        F.col("paymentType").cast("int").alias("payment_type"),
    )
    .withColumn("trip_minutes", (F.col("dropoff_ts").cast("long") - F.col("pickup_ts").cast("long")) / 60)
)

silver.write.mode("overwrite").format("delta").saveAsTable("taxi_silver")

print("Silver ridu:", spark.table("taxi_silver").count())
print("Bronze - Silver vahe:", spark.table("taxi_bronze").count() - spark.table("taxi_silver").count(), "rida filtreeriti välja")

In [ ]:
# Gold: päevane kokkuvõte äri mõõdikutega
gold = (
    spark.table("taxi_silver")
    .withColumn("pickup_date", F.to_date("pickup_ts"))
    .groupBy("pickup_date")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("total_amount"), 2).alias("avg_revenue_per_trip"),
        F.round(F.avg("trip_minutes"), 1).alias("avg_trip_minutes"),
        F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance"),
    )
    .orderBy("pickup_date")
)

gold.write.mode("overwrite").format("delta").saveAsTable("taxi_gold_daily")

display(spark.table("taxi_gold_daily"))

Kolm kihti annavad teineteisele hea puhvri:

- Bronze kihi ridade arv ja silver kihi ridade arv: vahe on kvaliteediindikaator. Kui äkki filtreerib välja 50% ridu, on midagi sisendis valesti.
- Gold kiht on see, mida Superset / Power BI / dashboard'id pärivad. Kiire ja vajadusel eel-agregeeritud.
- Kui nt silver kihi filter on liiga range, kusagil tekib mingi viga, vms, saad töövoo ümber kirjutada ilma bronze kihti uuesti allikast laadimata.

## 5. Mida Delta Lake annab võrreldes tavalise Parquet'iga

| Omadus | Parquet | Delta Lake |
|--------|---------|------------|
| Veerupõhine salvestus | jah | jah |
| Kompressioon | jah | jah |
| Skeema fail-tasemel | jah | jah |
| ACID transaktsioonid | ei | jah |
| UPDATE / DELETE / MERGE | ei (ainult ümberkirjutamine) | jah |
| Time travel | ei | jah |
| Skeemi evolutsioon (lisa veerg) | keeruline | jah (`mergeSchema`) |
| Failide kompaktimine | käsitsi | `OPTIMIZE` |
| Samaaegsete kirjutajate tugi | ei (risk) | jah (optimistic concurrency) |

**Millal valida Parquet Delta asemel?** Kui sa kirjutad andmed korra ja ei muuda kunagi (immutable archive). Kui andmed liiguvad edasi teise süsteemi ja Delta logi on tarbetu. Muudel juhtudel Delta on vaikimisi parem valik Databricksis.


Viide: [Delta Lake](https://delta.io/)

### Proovi ise

Lisa `taxi_silver` tabelile uus tuletatud veerg `is_airport_trip`: näiteks `trip_distance > 10` kui lihtne heuristika. Kasuta `ALTER TABLE` ja `UPDATE` või kirjuta tabel ümber koos uue veeruga.

Seejärel:

1. Loo uus gold tabel `taxi_gold_airport_daily`, mis grupeerib `pickup_date` ja `is_airport_trip` järgi.
2. Vaata `DESCRIBE HISTORY taxi_silver`: mitu versiooni nüüd on?
3. Tee `SELECT * FROM taxi_silver VERSION AS OF 0 LIMIT 5`: kas vana skeem on alles?

In [ ]:
# Sinu lahendus


## Kokkuvõte

Selles notebook'is sa:

- Lõid Delta tabeli ja tegid sellel UPDATE, DELETE ja MERGE operatsioonid
- Nägid `DESCRIBE HISTORY` abil iga operatsiooni jälje
- Lugesid varasema versiooni time travel'iga
- Jooksutasid `OPTIMIZE` ja said aru, miks `VACUUM` on eraldi samm
- Ehitasid mini medaljoni arhitektuuri (bronze → silver → gold) päris NYC Taxi andmetega

**Mida meeles pidada:**

- Delta = Parquet + `_delta_log`. ACID ja time travel tulevad logist.
- MERGE on tõeline game-changer andmetorustikes, kus tuleb andmeid upsert'ida.
- Medaljoni arhitektuur ei ole magic: see on lihtsalt kolm Delta tabelit, millest igaüks täidab kindlat ülesannet. Eraldus teeb hooldamise ja bug'ide parandamise palju lihtsamaks.
- Lakehouse'i peamine idee on, et sa saad data warehouse ACID garantiid ilma andmete kahekordse salvestamiseta kahes süsteemis.

## Praktikumi lõpp

Nüüd peaksid suutma:

- Näidata ühte enda kirjutatud DataFrame transformatsiooni ja selle SQL-ekvivalenti (notebook 01)
- Esitada EXPLAIN plaan, mis tõestab partition pruning töötamist (notebook 02)
- Näidata Delta tabelit vähemalt 3 versiooniga ja tuua esile varasem versioon (notebook 03)
- Selgitada medaljoni arhitektuuri ja põhjendada, miks kihid on eraldi (notebook 03)

Kui mõni samm ebaõnnestus, vaata [README veaotsingu](./README.md) osa.